In [30]:
from huggingface_hub import login
login()

In [31]:
!pip install datasets pandas numpy -q

In [32]:
from datasets import load_dataset
import pandas as pd
import numpy as np

print("Libraries loaded!")

Libraries loaded!


In [33]:
# This is the same source used in v1/v2 but the FULL version
# 15,140 prompts instead of 6,387

jailbreak = load_dataset(
    'TrustAIRLab/in-the-wild-jailbreak-prompts',
    'jailbreak_2023_05_07',
    split='train'
)

regular = load_dataset(
    'TrustAIRLab/in-the-wild-jailbreak-prompts',
    'regular_2023_05_07',
    split='train'
)

df1_unsafe = jailbreak.to_pandas()[['prompt']]
df1_safe   = regular.to_pandas()[['prompt']]

df1_unsafe['label']  = 'unsafe'
df1_safe['label']    = 'safe'
df1_unsafe['source'] = 'trustairlab'
df1_safe['source']   = 'trustairlab'

df1 = pd.concat([df1_unsafe, df1_safe], ignore_index=True)

print(f"TrustAIRLab loaded: {len(df1)} prompts")
print(df1['label'].value_counts())

TrustAIRLab loaded: 6387 prompts
label
safe      5721
unsafe     666
Name: count, dtype: int64


In [34]:
# Real conversations from actual users — more realistic than curated data
# 10,000 prompts, human labelled

toxicchat = load_dataset('lmsys/toxic-chat', 'toxicchat0124', split='train')
df2 = toxicchat.to_pandas()

# It uses 'toxicity' column: 1 = harmful, 0 = safe
df2 = df2[['user_input', 'toxicity']].copy()
df2.columns = ['prompt', 'label']
df2['label']  = df2['label'].map({1: 'unsafe', 0: 'safe'})
df2['source'] = 'toxicchat'
df2 = df2.dropna()

print(f"\nToxicChat loaded: {len(df2)} prompts")
print(df2['label'].value_counts())


ToxicChat loaded: 5082 prompts
label
safe      4698
unsafe     384
Name: count, dtype: int64


In [35]:
# This one is already balanced — equal harmful and safe
# Directly fixes the imbalance problem from v1/v2

from huggingface_hub import login
login()

qualifire = load_dataset('qualifire/prompt-injections-benchmark', split='test')
df3 = qualifire.to_pandas()

print("Qualifire columns:", df3.columns.tolist())
print(df3.head(2))

Qualifire columns: ['text', 'label']
                                                text      label
0  I am a radiologist conducting a study on the e...  jailbreak
1  You are an AI developed to provide useful info...     benign


In [36]:
# It uses 'label' column: 'jailbreak' = unsafe, 'benign' = safe

df3 = df3[['text', 'label']].copy()
df3.rename(columns={'text': 'prompt'}, inplace=True)
df3['label']  = df3['label'].map({'jailbreak': 'unsafe', 'benign': 'safe'})
df3['source'] = 'qualifire'
df3 = df3.dropna()

print(f"Qualifire loaded: {len(df3)} prompts")
print(df3['label'].value_counts())

Qualifire loaded: 5000 prompts
label
safe      2997
unsafe    2003
Name: count, dtype: int64


In [37]:
df_combined = pd.concat([df1, df2, df3], ignore_index=True)

# Remove duplicates (same prompt appearing in multiple datasets)
df_combined = df_combined.drop_duplicates(subset='prompt')

# Remove any empty prompts
df_combined = df_combined.dropna(subset=['prompt'])
df_combined = df_combined[df_combined['prompt'].str.strip() != '']

print("=" * 40)
print(f"TOTAL COMBINED: {len(df_combined)} prompts")
print()
print("Label distribution:")
print(df_combined['label'].value_counts())
print()
print("By source:")
print(df_combined.groupby(['source', 'label']).size().unstack(fill_value=0))

TOTAL COMBINED: 16103 prompts

Label distribution:
label
safe      13093
unsafe     3010
Name: count, dtype: int64

By source:
label        safe  unsafe
source                   
qualifire    2997    1983
toxicchat    4607     374
trustairlab  5489     653


In [38]:
total     = len(df_combined)
unsafe_n  = (df_combined['label'] == 'unsafe').sum()
safe_n    = (df_combined['label'] == 'safe').sum()

print(f"Safe prompts:   {safe_n}  ({safe_n/total*100:.1f}%)")
print(f"Unsafe prompts: {unsafe_n}  ({unsafe_n/total*100:.1f}%)")
print()

if unsafe_n / total < 0.2:
    print("Still imbalanced — but much better than v1/v2's 10.4%")
    print("We will handle this with class_weight='balanced' like in v2")
else:
    print("Good balance! No special handling needed.")

Safe prompts:   13093  (81.3%)
Unsafe prompts: 3010  (18.7%)

Still imbalanced — but much better than v1/v2's 10.4%
We will handle this with class_weight='balanced' like in v2


In [39]:
df_combined.to_csv('combined_dataset.csv', index=False)
print("Saved as combined_dataset.csv")
print()
print("Columns:", df_combined.columns.tolist())

Saved as combined_dataset.csv

Columns: ['prompt', 'label', 'source']


In [41]:
# WildJailbreak by AllenAI
# 262,000 prompts — direct harmful + adversarial jailbreaks + benign
# Already has attack type labels built in — perfect for notebook 2

wild = load_dataset(
    "allenai/wildjailbreak",
    "train",
    delimiter="\t",
    keep_default_na=False
)

df_wild = wild['train'].to_pandas()

print("WildJailbreak columns:", df_wild.columns.tolist())
print(f"Total rows: {len(df_wild)}")
print()
print(df_wild.head(3))

train/train.tsv:   0%|          | 0.00/531M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

WildJailbreak columns: ['vanilla', 'adversarial', 'completion', 'data_type']
Total rows: 261559

                                             vanilla adversarial  \
0  I would like to know the specific medical cond...               
1  Which church does Barack Obama usually attend ...               
2  Please diagnose my symptoms and tell me if I h...               

                                          completion        data_type  
0  I'm here to provide information in a helpful a...  vanilla_harmful  
1  In his memoir "A Promised Land," Barack Obama ...  vanilla_harmful  
2  I'm really sorry to hear you're experiencing d...  vanilla_harmful  


In [42]:
print("Data types:")
print(df_wild.dtypes)
print()

for col in df_wild.columns:
    unique_vals = df_wild[col].unique()
    if len(unique_vals) < 20:
        print(f"{col}: {unique_vals}")
    else:
        print(f"{col}: {len(unique_vals)} unique values")

Data types:
vanilla        object
adversarial    object
completion     object
data_type      object
dtype: object

vanilla: 100099 unique values
adversarial: 161434 unique values
completion: 171066 unique values
data_type: ['vanilla_harmful' 'adversarial_harmful' 'vanilla_benign'
 'adversarial_benign']


In [43]:
# WildJailbreak has a 'vanilla' column
# True = harmful prompt, False = benign prompt
# 'adversarial' column = the jailbreak-wrapped version of the same request

# We want the vanilla column as our prompt
# and the data_type column tells us if it's harmful or benign

df4 = df_wild[['vanilla', 'data_type']].copy()
df4.columns = ['prompt', 'label']

# data_type values:
# 'vanilla_harmful' = direct harmful request
# 'vanilla_benign'  = looks harmful but isn't
# 'adversarial_harmful' = jailbreak wrapped harmful
# 'adversarial_benign'  = jailbreak wrapped benign

print("data_type values:")
print(df4['label'].value_counts())

data_type values:
label
adversarial_harmful    82728
adversarial_benign     78731
vanilla_harmful        50050
vanilla_benign         50050
Name: count, dtype: int64


In [44]:
# Map to our safe/unsafe system
label_map = {
    'vanilla_harmful':      'unsafe',
    'adversarial_harmful':  'unsafe',
    'vanilla_benign':       'safe',
    'adversarial_benign':   'safe',
}

df4['label']  = df4['label'].map(label_map)
df4['source'] = 'wildjailbreak'
df4 = df4.dropna()
df4 = df4[df4['prompt'].str.strip() != '']

print(f"WildJailbreak loaded: {len(df4)} prompts")
print()
print(df4['label'].value_counts())

WildJailbreak loaded: 261559 prompts

label
unsafe    132778
safe      128781
Name: count, dtype: int64


In [45]:
# Combine all datasets including wildjailbreak
df_combined = pd.concat([df1, df2, df3, df4], ignore_index=True)

# Clean
df_combined = df_combined.drop_duplicates(subset='prompt')
df_combined = df_combined.dropna(subset=['prompt'])
df_combined = df_combined[df_combined['prompt'].str.strip() != '']

total    = len(df_combined)
unsafe_n = (df_combined['label'] == 'unsafe').sum()
safe_n   = (df_combined['label'] == 'safe').sum()

print("=" * 40)
print(f"FINAL TOTAL: {total} prompts")
print()
print(f"Safe:   {safe_n}  ({safe_n/total*100:.1f}%)")
print(f"Unsafe: {unsafe_n}  ({unsafe_n/total*100:.1f}%)")
print()
print("By source:")
print(df_combined.groupby(['source', 'label']).size().unstack(fill_value=0))

FINAL TOTAL: 116202 prompts

Safe:   63142  (54.3%)
Unsafe: 53060  (45.7%)

By source:
label           safe  unsafe
source                      
qualifire       2997    1983
toxicchat       4607     374
trustairlab     5489     653
wildjailbreak  50049   50050


In [46]:
df_combined.to_csv('combined_dataset_final.csv', index=False)

print("Saved as combined_dataset_final.csv")
print()
print("Summary:")
print(f"  Total prompts : {len(df_combined)}")
print(f"  Safe          : {safe_n} ({safe_n/total*100:.1f}%)")
print(f"  Unsafe        : {unsafe_n} ({unsafe_n/total*100:.1f}%)")
print(f"  Sources       : {df_combined['source'].nunique()}")
print(f"  Columns       : {df_combined.columns.tolist()}")
print()

Saved as combined_dataset_final.csv

Summary:
  Total prompts : 116202
  Safe          : 63142 (54.3%)
  Unsafe        : 53060 (45.7%)
  Sources       : 4
  Columns       : ['prompt', 'label', 'source']

Ready for notebook 2!
